In [1]:
import numpy as np
import matplotlib.pyplot as plt

from models.LSTM_AE import LSTMAutoencoder

import torch
import torch.nn as nn
from torch.nn import MSELoss
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from tqdm import tqdm

In [2]:
data_rec = np.load('../../data/Lorenz/lorenz_data_rec0_1,0_0,1_1.npy')
test_rec = np.load('../../data/Lorenz/lorenz_data_rec_test0_1,0_0,1_1.npy')
print(data_rec.shape)
class LorenzDataset(Dataset):
    def __init__(self, data_rec):
        self.data_rec = torch.from_numpy(data_rec)

    def __len__(self):
        return len(self.data_rec)

    def __getitem__(self, idx):
        return self.data_rec[idx]

dataset = LorenzDataset(data_rec)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

test_dataset = LorenzDataset(test_rec)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=True)

(17400, 40, 3)


In [3]:
device = torch.device("cuda:0")
print(device)
print(torch.cuda.get_device_name(0))

cuda:0
NVIDIA GeForce RTX 4070 Ti SUPER


In [10]:

model_rec = LSTMAutoencoder(input_dim=3, hidden_dim=16, num_layers=1).to(device)
criterion = nn.MSELoss()
optimizer = Adam(model_rec.parameters(), lr=1e-3)

num_epochs = 100
patience = 10   # сколько эпох ждать улучшения
best_loss = float("inf")
patience_counter = 0

train_losses = []
test_losses = []

for epoch in range(num_epochs):
    # === TRAIN ===
    model_rec.train()
    running_loss = 0.0
    for x in tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]"):
        x = x.to(device).float()

        optimizer.zero_grad()
        y, _ = model_rec(x)
        loss = criterion(y, x)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)

    epoch_train_loss = running_loss / len(dataloader.dataset)
    train_losses.append(epoch_train_loss)

    # === VALIDATION ===
    model_rec.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x in tqdm(test_dataloader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]"):
            x = x.to(device).float()
            y, _ = model_rec(x)
            loss = criterion(y, x)
            val_loss += loss.item() * x.size(0)

    epoch_val_loss = val_loss / len(test_dataloader.dataset)
    test_losses.append(epoch_val_loss)

    print(f"Epoch {epoch+1}: train_loss={epoch_train_loss:.6f}, val_loss={epoch_val_loss:.6f}")

    # === Early stopping ===
    if epoch_val_loss < best_loss:
        best_loss = epoch_val_loss
        patience_counter = 0
        torch.save(model_rec.state_dict(), "best_lstm_ae.pth")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

# === Plot losses ===
plt.figure(figsize=(8,5))
plt.plot(train_losses, label="Train loss")
plt.plot(test_losses, label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("LSTM-AE Training")
plt.show()

AssertionError: Torch not compiled with CUDA enabled

In [10]:
# === Load best model ===
model_rec.load_state_dict(torch.load("best_lstm_ae.pth"))
model_rec.eval()

KeyboardInterrupt: 